In [ ]:
!pip install -qU transformers accelerate pillow pandas torch

In [1]:
!pip uninstall -y transformers
!pip install transformers==4.43.3

Found existing installation: transformers 5.0.0
Uninstalling transformers-5.0.0:
  Successfully uninstalled transformers-5.0.0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.4/9.4 MB 73.5 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 32.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 101.3 MB/s eta 0:00:0000:01
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.4.1
    Uninstalling huggingface_hub-1.4.1:
      Successfully uninstalled huggingface_hub-1.4.1
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.2
    Uninstalling tokenizers-0.22.2:
      Successfully uninstalled tokenizers-0.22.2


In [2]:
import pandas as pd
import torch
from PIL import Image
from transformers import AutoModelForCausalLM, AutoProcessor
import os

# Define the model ID
model_id = "microsoft/Phi-3-vision-128k-instruct"

print("Loading processor...")
processor = AutoProcessor.from_pretrained(model_id, trust_remote_code=True)

print("Loading model (this may take a few minutes)...")
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto", # Automatically assigns model to available GPU(s)
    trust_remote_code=True,
    torch_dtype=torch.float16, # Uses 16-bit precision to save Kaggle GPU memory
    _attn_implementation='eager' # Standard attention, stable on Kaggle T4s
)
print("Model loaded successfully!")

2026-04-20 17:43:15.620454: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776706995.848132      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776706995.919163      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776706996.454134      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776706996.454172      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776706996.454175      55 computation_placer.cc:177] computation placer alr

Loading processor...


preprocessor_config.json:   0%|          | 0.00/464 [00:00<?, ?B/s]

processing_phi3_v.py: 0.00B [00:00, ?B/s]

image_processing_phi3_v.py: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.94G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/3.35G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Model loaded successfully!


In [4]:
IMAGE_PATH="/kaggle/input/datasets/shreyas0s/omnigen2-evaluation-images/Evaluation Images/omnigen2/"
csv_path = "/kaggle/input/datasets/shreyas0s/omnigen2-evaluation-images/Evaluation Images/evaluation_data.csv"
df = pd.read_csv(csv_path)

sample_df = df

results = []

for index, row in sample_df.iterrows():
    image_path = IMAGE_PATH + str(row['image_name']) + ".png"
    caption = row['description']
    
    print(f"\n--- Processing Row {index} ---")
    print(f"Caption: {caption}")
    
    # 2. Check if image exists and load it
    if not os.path.exists(image_path):
        print(f"Error: Image not found at {image_path}. Skipping.")
        continue
        
    try:
        # Convert to RGB to avoid issues with PNGs having alpha channels
        image = Image.open(image_path).convert("RGB") 
    except Exception as e:
        print(f"Error loading image: {e}")
        continue

    # 3. Construct the prompt using Phi-3's chat template
    # Phi-3-vision requires the <|image_1|> tag to know where the image belongs
    user_prompt = (
        f"<|image_1|>\n"
        f"The intended caption for this image is: '{caption}'.\n"
        f"Analyze the image carefully. What specific improvements, additions, "
        f"or changes should be made to this image so that it perfectly matches the given caption?"
    )
    
    messages = [
        {"role": "user", "content": user_prompt}
    ]
    
    # Apply chat template
    prompt = processor.tokenizer.apply_chat_template(
        messages, 
        tokenize=False, 
        add_generation_prompt=True
    )
    
    # 4. Prepare inputs for the model
    inputs = processor(prompt, [image], return_tensors="pt").to("cuda")
    
    # 5. Generate the response
    generation_args = {
        "max_new_tokens": 300, # Adjust based on how long you want the explanation to be
        "temperature": 0.1,    # Keep low for more analytical, less creative responses
        "do_sample": False,
    }
    
    with torch.no_grad():
        generate_ids = model.generate(
            **inputs, 
            eos_token_id=processor.tokenizer.eos_token_id, 
            **generation_args
        )
    
    # 6. Decode the output (removing the input prompt from the output text)
    # Slice the generated ids to exclude the input tokens
    generate_ids = generate_ids[:, inputs['input_ids'].shape[1]:]
    response = processor.batch_decode(
        generate_ids, 
        skip_special_tokens=True, 
        clean_up_tokenization_spaces=False
    )[0]
    
    print(f"Suggested Improvements:\n{response}")
    
    # Store results if you want to save them back to a CSV later
    results.append({
        "image_path": image_path,
        "caption": caption,
        "suggested_improvements": response
    })
# Optional: Save results to a new CSV
# results_df = pd.DataFrame(results)
# results_df.to_csv("image_improvements.csv", index=False)
# print("\nSaved results to image_improvements.csv")


--- Processing Row 0 ---
Caption: Figure 5.11 illustrates racemose inflorescence, an indeterminate flowering pattern characterized by continued apical growth of the main axis while flowers develop acropetally along lateral branches. The specimen displays numerous yellow flowers in various developmental stages—from mature open flowers to unopened buds—arranged along primary and secondary axes. The compound pinnate leaves at the stem base provide photosynthetic support. This branching architecture, where the peduncle elongates indefinitely and pedicels bear individual flowers, demonstrates typical racemose characteristics with older flowers positioned basally and progressively younger buds toward the apex.


/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:567: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.1` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/root/.cache/huggingface/modules/transformers_modules/microsoft/Phi-3-vision-128k-instruct/ed2772fabe9dc9acd0caad54b62761d92520cc44/image_embedding_phi3_v.py:197: UserWarning: Phi-3-V modifies `input_ids` in-place and the tokens indicating images will be removed after model forward. If your workflow requires multiple forward passes on the same `input_ids`, please make a copy of `input_ids` before passing it to the model.
  warnings.warn(


Suggested Improvements:
To match the given caption, the image should be enhanced to clearly show the indeterminate flowering pattern of the racemose inflorescence. This could involve digitally adding more flowers in various stages of development along the main axis, ensuring they are arranged in a way that demonstrates the characteristic of older flowers being positioned basally and progressively younger buds toward the apex. Additionally, the image could be improved by increasing the contrast between the flowers and the background, and by adding a clearer view of the compound pinnate leaves at the stem base to emphasize their role in photosynthetic support.

--- Processing Row 1 ---
Caption: This anatomical model displays the heart's four-chambered structure with color-coded blood flow pathways. The blue vessels carry deoxygenated blood into the heart, while red vessels transport oxygenated blood. The interior reveals muscular chambers separated by walls, with the yellow lines demarca

In [5]:
import json

In [6]:
output_filename = "image_improvements.json"

with open(output_filename, "w", encoding="utf-8") as f:
    # indent=4 makes the JSON nicely formatted and readable by humans
    json.dump(results, f, indent=4, ensure_ascii=False)

print(f"\nSaved results to {output_filename}")


Saved results to image_improvements.json
